# Two-minus-sector amplitude $A_n$ — closed form & verification

1D deep-water surface waves, sector $\sigma=(-1,-1,+1,\dots,+1)$.

$$A_n = 2^{\,n-1}\, i\, \omega_1\, \omega_2^{\,2n-5}\,/\,g^{\,n-3}$$

where $\omega_1,\omega_2$ are the two $\sigma=-1$ legs.  Valid in the **principal
chamber** $|\omega_2|=\min_i|\omega_i|$ (all sorted positive-frequency kinematics).

This notebook checks it against an **independent** exact-rational Berends–Giele
recursion (`waterwave_bg.py`), which itself reproduces the supplied Mathematica
`BGAmplitude` bit-for-bit.

In [1]:
from fractions import Fraction as F
from waterwave_bg import (BGAmplitude, make_kinematics, two_minus_sigma,
                          closed_form, GR)

def check(n, freeW, g=1):
    ks, ws = make_kinematics(n, freeW, two_minus_sigma(n), g)
    bg = BGAmplitude(ks, ws, g)
    cf = closed_form(n, ws, g)
    print(f"n={n} free={[str(x) for x in freeW]}")
    print(f"   w        = {[str(x) for x in ws]}")
    print(f"   BG       = {bg}")
    print(f"   formula  = {cf}")
    print(f"   EXACT match: {bg == cf}\n")
    return bg == cf

## n = 5, 6, 7 in the principal chamber (incl. extreme hierarchies)

In [2]:
ok = []
ok.append(check(5, [F(3,2), 2, F(5,2)]))
ok.append(check(5, [2, 3, 7]))
ok.append(check(5, [1, 10, 100]))          # huge plus legs
ok.append(check(5, [F(1,1000), 1, 1]))     # tiny w2
ok.append(check(6, [F(3,2), 2, F(5,2), 3]))
ok.append(check(6, [1, 3, 5, 7]))
ok.append(check(7, [F(3,2), 2, F(5,2), 3, F(7,2)]))
ok.append(check(7, [F(1,10), 2, 4, 8, 16]))
print('ALL EXACT:', all(ok))

n=5 free=['3/2', '2', '5/2']
   w        = ['-11/3', '3/2', '2', '5/2', '-7/3']
   BG       = (0 + -891/2 i)
   formula  = (0 + -891/2 i)
   EXACT match: True

n=5 free=['2', '3', '7']
   w        = ['-33/4', '2', '3', '7', '-15/4']
   BG       = (0 + -4224 i)
   formula  = (0 + -4224 i)
   EXACT match: True

n=5 free=['1', '10', '100']
   w        = ['-11210/111', '1', '10', '100', '-1111/111']
   BG       = (0 + -179360/111 i)
   formula  = (0 + -179360/111 i)
   EXACT match: True

n=5 free=['1/1000', '1', '1']
   w        = ['-3002/2001', '1/1000', '1', '1', '-1002001/2001000']
   BG       = (0 + -1501/62531250000000000 i)
   formula  = (0 + -1501/62531250000000000 i)
   EXACT match: True

n=6 free=['3/2', '2', '5/2', '3']
   w        = ['-49/9', '3/2', '2', '5/2', '3', '-32/9']
   BG       = (0 + -11907/4 i)
   formula  = (0 + -11907/4 i)
   EXACT match: True

n=6 free=['1', '3', '5', '7']
   w        = ['-169/16', '1', '3', '5', '7', '-87/16']
   BG       = (0 + -338 i)
   formula

## g ≠ 1 (restore gravitational acceleration)

In [3]:
for g in [F(2), F(7,3), F(981,100)]:
    check(5, [F(3,2), 2, F(5,2)], g=g)
    check(6, [1, 3, 5, 7], g=g)

n=5 free=['3/2', '2', '5/2']
   w        = ['-11/3', '3/2', '2', '5/2', '-7/3']
   BG       = (0 + -891/8 i)
   formula  = (0 + -891/8 i)
   EXACT match: True

n=6 free=['1', '3', '5', '7']
   w        = ['-169/16', '1', '3', '5', '7', '-87/16']
   BG       = (0 + -169/4 i)
   formula  = (0 + -169/4 i)
   EXACT match: True

n=5 free=['3/2', '2', '5/2']
   w        = ['-11/3', '3/2', '2', '5/2', '-7/3']
   BG       = (0 + -8019/98 i)
   formula  = (0 + -8019/98 i)
   EXACT match: True

n=6 free=['1', '3', '5', '7']
   w        = ['-169/16', '1', '3', '5', '7', '-87/16']
   BG       = (0 + -9126/343 i)
   formula  = (0 + -9126/343 i)
   EXACT match: True

n=5 free=['3/2', '2', '5/2']
   w        = ['-11/3', '3/2', '2', '5/2', '-7/3']
   BG       = (0 + -55000/11881 i)
   formula  = (0 + -55000/11881 i)
   EXACT match: True

n=6 free=['1', '3', '5', '7']
   w        = ['-169/16', '1', '3', '5', '7', '-87/16']
   BG       = (0 + -338000000/944076141 i)
   formula  = (0 + -338000000/9440761

## n = 4 is degenerate — verified as a limit

On-shell forces $\omega_4=-\omega_2$ (a zero-momentum/zero-frequency pair), so BG
is $0/0$ at the exact point.  Detune $\omega_4=-\omega_2+t$ and let $t\to0$:

In [4]:
def A4_detuned(w2, w3, t, g=1):
    w4 = -w2 + t; w1 = -(w2 + w3 + w4)
    ws = [w1, w2, w3, w4]
    ks = [s*w*w/F(g) for s, w in zip([-1,-1,1,1], ws)]
    return BGAmplitude(ks, ws, g)

w2, w3 = F(3,2), F(2)
print('predicted A_4 = 8 i w1 w2^3, w1->-w3 :', complex(0, 8*float(-w3)*float(w2)**3))
for t in [F(1,10), F(1,100), F(1,1000), F(1,10**6), F(1,10**9)]:
    a = A4_detuned(w2, w3, t)
    print(f'  t={float(t):.0e}  A_4 = {float(a.re):+.6f} {float(a.im):+.9f} i')

predicted A_4 = 8 i w1 w2^3, w1->-w3 : -54j
  t=1e-01  A_4 = +0.000000 -49.858100000 i
  t=1e-02  A_4 = +0.000000 -53.609803010 i
  t=1e-03  A_4 = +0.000000 -53.961223003 i
  t=1e-06  A_4 = +0.000000 -53.999961250 i
  t=1e-09  A_4 = +0.000000 -53.999999961 i


## Outside the principal chamber the monomial does NOT hold (other waterhedron chambers)

In [5]:
# w2 largest -> chamber-B form is 32 i w1 w2 w3^2 w4^2, not the principal monomial
ks, ws = make_kinematics(5, [1000, 1, 1], two_minus_sigma(5), 1)
bg = BGAmplitude(ks, ws, 1)
mono = closed_form(5, ws, 1)
chamberB = GR(0,1)*32*ws[0]*ws[1]*ws[2]**2*ws[3]**2
print('w =', [str(x) for x in ws], '  (|w2| is NOT the smallest)')
print('BG                                  =', bg)
print('principal monomial                  =', mono, '  match:', bg==mono)
print('chamber-B form 32 i w1 w2 w3^2 w4^2  =', chamberB, '  match:', bg==chamberB)

w = ['-2003/1002', '1000', '1', '1', '-1002001/1002']   (|w2| is NOT the smallest)
BG                                  = (0 + -32048000/501 i)
principal monomial                  = (0 + -16024000000000000000/501 i)   match: False
chamber-B form 32 i w1 w2 w3^2 w4^2  = (0 + -32048000/501 i)   match: True
